# Практическая работа 3. Часть 2 — предобработка данных

**Датасет:** IBM HR Analytics Employee Attrition & Performance (Kaggle).
**Целевая переменная:** `Attrition` (уволился / нет).

**План**
1. Загрузка и та же очистка, что в части 1 (блокнот должен работать самостоятельно).
2. Разделение на train и test (80/20).
3. Один числовой признак — `MonthlyIncome`: несколько методов масштабирования, графики до и после, выбор метода.
4. Предобработка всех остальных столбцов с обоснованием выбора метода.

Выполнил: Гегечкори Георгий

## 1. Загрузка данных и библиотек

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, MaxAbsScaler, RobustScaler,
                                   OneHotEncoder, OrdinalEncoder, LabelEncoder, FunctionTransformer)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", 60)

In [ ]:
import os
FILE = "WA_Fn-UseC_-HR-Employee-Attrition.csv"
# файл ищем рядом с блокнотом или в папке data/
PATH = next((p for p in [FILE, os.path.join("data", FILE)] if os.path.exists(p)), None)

if PATH is None:
    # в Colab файла рядом нет — скачиваем датасет с Kaggle (как в ПР2)
    !pip install -q opendatasets
    import opendatasets as od
    od.download("https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset")
    PATH = "ibm-hr-analytics-attrition-dataset/WA_Fn-UseC_-HR-Employee-Attrition.csv"

df_raw = pd.read_csv(PATH)
df_raw.shape

## 2. Очистка данных

Повторяем очистку из части 1 (подробное обоснование каждого шага — там):
- дубликатов и пропусков в исходных данных нет — проверяем ещё раз;
- удаляем константные столбцы `EmployeeCount`, `StandardHours`, `Over18` и идентификатор `EmployeeNumber`;
- удаляем `DailyRate`, `HourlyRate`, `MonthlyRate` — они не согласуются с доходом (корреляция ≈ 0);
- логически невозможные значения `NumCompaniesWorked` (0 компаний при наличии текущей работы и 1 компания при стаже
  до текущей) заменяем на NaN — заполним их после разделения на train и test.

In [ ]:
df = df_raw.copy()
print("Полных копий:", df.duplicated().sum(), "| Пропусков:", df.isna().sum().sum())

df = df.drop_duplicates()
df = df.drop(columns=["EmployeeCount", "StandardHours", "Over18", "EmployeeNumber",
                      "DailyRate", "HourlyRate", "MonthlyRate"])

prev_years = df["TotalWorkingYears"] - df["YearsAtCompany"]
bad_ncw = (df["NumCompaniesWorked"] == 0) | ((df["NumCompaniesWorked"] == 1) & (prev_years > 0))
df.loc[bad_ncw, "NumCompaniesWorked"] = np.nan

print("Размер после очистки:", df.shape)
print("NaN в NumCompaniesWorked:", df["NumCompaniesWorked"].isna().sum())

## 3. Разделение на train и test (80/20)

Разделяем **до** любых преобразований, которые что-то «запоминают» о данных (среднее, медиану, минимум и максимум, список категорий).
Если считать их по всей таблице, информация из test попадёт в обучение — это утечка данных, и оценка модели на test
окажется завышенной.

- `test_size=0.2` — 80 % на обучение, 20 % на проверку;
- `stratify=y` — в обеих частях одинаковая доля уволившихся (≈16 %). Без этого при несбалансированных классах
  в test могло бы случайно попасть заметно больше или меньше уволившихся;
- `random_state=42` — разбиение воспроизводимо.

In [ ]:
X = df.drop(columns="Attrition")
y_raw = df["Attrition"]

X_train, X_test, y_train_raw, y_test_raw = train_test_split(
    X, y_raw, test_size=0.2, stratify=y_raw, random_state=42)

print("train:", X_train.shape, "| test:", X_test.shape)
print("Доля Yes: train = {:.1%}, test = {:.1%}".format((y_train_raw == "Yes").mean(), (y_test_raw == "Yes").mean()))

**Целевая переменная.** `Attrition` — текстовые метки Yes/No. Для целевой переменной предназначен `LabelEncoder`:
он превращает классы в числа 0 и 1. Числа здесь — просто номера классов, а не порядок или расстояние.
Кодировщик обучаем на train и тем же кодировщиком переводим test.

In [ ]:
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test = le.transform(y_test_raw)
dict(zip(le.classes_, le.transform(le.classes_)))

## 4. Масштабирование одного признака: `MonthlyIncome`

**Почему этот признак.** Месячный доход измеряется в денежных единицах от ≈1000 до ≈20000, тогда как большинство других
признаков — это шкалы 1–4 или годы стажа. Кроме того, распределение сильно скошено вправо и содержит много выбросов —
высокие зарплаты руководителей (в ПР2 мы нашли 114 таких сотрудников и решили их не удалять: это не ошибки,
а реальная и важная для задачи группа). Поэтому на доходе хорошо видно, чем методы масштабирования отличаются друг от друга.

Все преобразования **обучаем на train** (`fit_transform`), а к test применяем уже найденные параметры (`transform`).

In [ ]:
income_train = X_train[["MonthlyIncome"]]
income_test = X_test[["MonthlyIncome"]]

methods = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "MaxAbsScaler": MaxAbsScaler(),
    "RobustScaler": RobustScaler(),
    "log1p": FunctionTransformer(np.log1p),
    "log1p + StandardScaler": Pipeline([("log", FunctionTransformer(np.log1p)), ("std", StandardScaler())]),
}

scaled_train = {"Исходный": income_train["MonthlyIncome"].to_numpy()}
scaled_test = {"Исходный": income_test["MonthlyIncome"].to_numpy()}
for name, transformer in methods.items():
    scaled_train[name] = np.asarray(transformer.fit_transform(income_train)).ravel()   # параметры — только по train
    scaled_test[name] = np.asarray(transformer.transform(income_test)).ravel()       # те же параметры для test

### 4.1 Графики распределения до и после каждого преобразования (train)

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(13, 15))
axes = axes.ravel()
for ax, (name, values) in zip(axes, scaled_train.items()):
    ax.hist(values, bins=30, edgecolor="black", color="#DD8452" if name == "Исходный" else "#4C72B0")
    ax.axvline(np.median(values), color="black", linestyle="--", label=f"медиана = {np.median(values):.2f}")
    ax.axvline(np.mean(values), color="red", linestyle=":", label=f"среднее = {np.mean(values):.2f}")
    ax.set_title(name)
    ax.set_ylabel("Количество сотрудников")
    ax.legend(fontsize=8)

# последняя ячейка: все методы на одной оси в виде boxplot — видно, где оказываются выбросы
linear = ["StandardScaler", "MinMaxScaler", "MaxAbsScaler", "RobustScaler", "log1p + StandardScaler"]
axes[-1].boxplot([scaled_train[m] for m in linear])
axes[-1].set_xticks(range(1, len(linear) + 1), ["Standard", "MinMax", "MaxAbs", "Robust", "log1p+Std"], fontsize=8)
axes[-1].axhline(0, color="grey", linewidth=0.8)
axes[-1].set_title("Сравнение шкал (boxplot)")

plt.suptitle("MonthlyIncome: распределение до и после преобразований (train)", fontsize=14)
plt.tight_layout()
plt.show()

### 4.2 Числовые характеристики после преобразований

In [ ]:
def describe(values):
    s = pd.Series(values)
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return {
        "среднее": s.mean(), "ст. откл.": s.std(), "медиана": s.median(),
        "IQR": iqr, "мин": s.min(), "макс": s.max(),
        "асимметрия": s.skew(),
        "выбросов (1.5·IQR)": int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()),
    }

stats_table = pd.DataFrame({name: describe(v) for name, v in scaled_train.items()}).T
stats_table.round(3)

In [ ]:
# MinMaxScaler: где оказываются сотрудники на отрезке [0, 1]
mm = pd.Series(scaled_train["MinMaxScaler"])
print("Квартили после MinMax:", mm.quantile([0.25, 0.5, 0.75]).round(3).tolist())
print("Доля сотрудников в нижней половине шкалы (< 0.5): {:.1%}".format((mm < 0.5).mean()))

In [ ]:
# test преобразован параметрами train — значения могут слегка выходить за «идеальные» границы, и это нормально
pd.DataFrame({name: {"мин (test)": v.min(), "макс (test)": v.max()} for name, v in scaled_test.items()}).T.round(3)

### 4.3 Вывод о методах

**Что общего у линейных методов** (Standard, MinMax, MaxAbs, Robust). Они только сдвигают и растягивают шкалу.
Форма распределения не меняется: на всех четырёх гистограммах тот же длинный правый хвост, асимметрия остаётся 1,33,
число выбросов по правилу 1,5·IQR — те же 86 на train. Как и говорилось в теории: масштабирование меняет координаты, а не наблюдения.

**Различия между ними:**
- **StandardScaler** — среднее 0, стандартное отклонение 1. Но среднее и отклонение сами «перетянуты» высокими зарплатами:
  медиана оказывается не в нуле, а около −0,33, а самые высокие доходы — на уровне ≈ +2,9.
- **MinMaxScaler** — шкала [0, 1] по минимуму и максимуму train. Максимум — это как раз выброс, поэтому центральные 50 % сотрудников
  попадают в полосу 0,10–0,39, а 83 % всех сотрудников — в нижнюю половину шкалы. Самый чувствительный к выбросам метод.
- **MaxAbsScaler** — делит на максимум модуля. Для положительного признака это почти то же, что MinMax, только минимум не уходит в 0.
  Метод создан для разреженных данных с большим числом нулей — у дохода нулей нет, поэтому преимуществ он здесь не даёт.
- **RobustScaler** — медиана становится 0, масштаб задаёт межквартильный размах. Центральная часть распределения не зависит
  от выбросов, но сами выбросы остаются далеко (до ≈ +2,7).
- **log1p** — нелинейное преобразование: сильнее сжимает большие значения. Асимметрия падает с 1,33 до 0,25,
  выбросов по правилу 1,5·IQR не остаётся (было 86), распределение становится близким к симметричному.
  Высокие зарплаты при этом **сохраняются** — они просто перестают доминировать над остальными.
- **log1p + StandardScaler** — сначала убираем асимметрию, затем приводим к среднему 0 и отклонению 1.

**Какой метод лучше для `MonthlyIncome`.** Для этого признака лучше всего подходит **log1p + StandardScaler**:
логарифм решает главную проблему признака — правую асимметрию и выбросы, — а стандартизация приводит его к той же шкале,
что и остальные числовые признаки. Это особенно важно для логистической регрессии, которую мы планировали в ПР1:
её коэффициенты зависят от масштаба, а выбросы на линейной шкале сильно на неё влияют.
Если бы логарифм по какой-то причине был недопустим (например, признак мог быть отрицательным), следующим по качеству был бы
**RobustScaler** — он не зависит от выбросов. MinMaxScaler для этого признака — худший выбор.
Для деревьев решений и случайного леса масштабирование не требуется вовсе — они делят признак по порогам.

## 5. Предобработка всех столбцов

Разные типы столбцов требуют разных преобразований. Сначала разберём столбцы по группам, затем соберём единую схему
`ColumnTransformer` и обучим её **только на train**.

In [ ]:
X_train.dtypes.value_counts()

In [ ]:
for col in X_train.select_dtypes(exclude="number").columns:
    print(f"{col:15s}", sorted(X_train[col].unique()))

### 5.1 Порядковые шкалы

Эти признаки уже числовые, но это не «настоящие» числа, а **порядковые оценки** с известными границами:
удовлетворённость и вовлечённость 1–4, образование 1–5, уровень должности 1–5, опционы 0–3, оценка эффективности 3–4.
Порядок у них есть (4 лучше 1), поэтому кодировать заново не нужно. Выбросов у них не бывает — значения ограничены шкалой.

**Метод: MinMaxScaler** — переводит каждую шкалу в [0, 1] с сохранением порядка, и шкалы разной длины (1–4 и 1–5)
становятся сопоставимы. Здесь MinMax уместен, потому что минимум и максимум — это границы шкалы, а не выбросы.

In [ ]:
ordinal_scales = ["Education", "EnvironmentSatisfaction", "JobInvolvement", "JobLevel", "JobSatisfaction",
                  "PerformanceRating", "RelationshipSatisfaction", "StockOptionLevel", "WorkLifeBalance"]
X_train[ordinal_scales].agg(["min", "max", "nunique"])

### 5.2 Непрерывные числовые признаки

Остальные числовые признаки — годы, расстояние, доход, проценты. Как показал разбор `MonthlyIncome`, для скошенного
признака лучше сначала взять логарифм. Чтобы не решать «на глаз», считаем асимметрию **на train** и делим признаки на две группы:
- асимметрия > 0,75 — сильно скошенные: **log1p + StandardScaler** (обоснование — в разделе 4.3);
- остальные — близкие к симметричным: достаточно **StandardScaler**.

Все значения неотрицательные, поэтому log1p (логарифм от 1 + x) применим, в том числе к нулям стажа.

**Пропуски.** В `NumCompaniesWorked` есть пропуски — 244 логические ошибки из части 1, распределённые между train и test. Заполняем их **медианой train** (`SimpleImputer`):
медиана устойчива к выбросам, а считается только по обучающей части, чтобы не было утечки.
Импутер ставим первым шагом в обе ветки — в остальных столбцах пропусков нет, и он их не изменит.

In [ ]:
continuous = [c for c in X_train.select_dtypes(include="number").columns if c not in ordinal_scales]
skew = X_train[continuous].skew().sort_values(ascending=False)
skew.round(2).to_frame("асимметрия (train)")

In [ ]:
skewed = skew[skew > 0.75].index.tolist()
symmetric = skew[skew <= 0.75].index.tolist()
print("log1p + StandardScaler:", skewed)
print("StandardScaler:        ", symmetric)

### 5.3 Категориальные признаки

- **BusinessTravel** — `Non-Travel` → `Travel_Rarely` → `Travel_Frequently`: у категорий есть естественный порядок
  (частота командировок растёт). Поэтому **OrdinalEncoder с явно заданным порядком** 0 → 1 → 2, затем MinMaxScaler в [0, 1].
  Порядок задаём вручную: по умолчанию кодировщик расставил бы категории по алфавиту и нарушил смысл.
- **Gender, OverTime** — бинарные признаки. **OneHotEncoder(drop="if_binary")** даёт один столбец 0/1:
  второй был бы полной копией первого «наоборот».
- **Department, EducationField, JobRole, MaritalStatus** — номинальные, порядка нет (нельзя сказать, что «Sales» больше
  «Research & Development»). Коды 0, 1, 2 создали бы ложную числовую шкалу, поэтому **OneHotEncoder**: каждая категория —
  отдельный столбец 0/1. `handle_unknown="ignore"` — если в новых данных появится категория, которой не было в train,
  она закодируется нулевым вектором, а не вызовет ошибку.
  Уникальных значений немного (3–9), поэтому объединять редкие категории не нужно.

In [ ]:
travel_order = [["Non-Travel", "Travel_Rarely", "Travel_Frequently"]]
binary_cats = ["Gender", "OverTime"]
nominal_cats = ["Department", "EducationField", "JobRole", "MaritalStatus"]

### 5.4 Единая схема и обучение на train

In [ ]:
preprocessor = ColumnTransformer([
    ("skewed", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scale", StandardScaler()),
    ]), skewed),
    ("symmetric", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), symmetric),
    ("ordinal_scales", MinMaxScaler(), ordinal_scales),
    ("travel", Pipeline([
        ("encode", OrdinalEncoder(categories=travel_order)),
        ("scale", MinMaxScaler()),
    ]), ["BusinessTravel"]),
    ("binary", OneHotEncoder(drop="if_binary"), binary_cats),
    ("nominal", OneHotEncoder(handle_unknown="ignore"), nominal_cats),
], verbose_feature_names_out=False)

preprocessor

In [ ]:
# fit_transform — только на train: здесь вычисляются медианы, средние, отклонения, min/max и списки категорий
X_train_prep = pd.DataFrame(preprocessor.fit_transform(X_train),
                            columns=preprocessor.get_feature_names_out(), index=X_train.index)

# transform — для test: применяются параметры, найденные на train
X_test_prep = pd.DataFrame(preprocessor.transform(X_test),
                           columns=preprocessor.get_feature_names_out(), index=X_test.index)

print("train:", X_train.shape, "→", X_train_prep.shape)
print("test: ", X_test.shape, "→", X_test_prep.shape)
print("Пропусков после обработки: train =", X_train_prep.isna().sum().sum(), "| test =", X_test_prep.isna().sum().sum())

In [ ]:
X_train_prep.head()

### 5.5 Проверка результата

In [ ]:
# медиана для заполнения NumCompaniesWorked взята из train
imputer = preprocessor.named_transformers_["skewed"].named_steps["impute"]
dict(zip(skewed, imputer.statistics_))

In [ ]:
# на train стандартизованные признаки имеют среднее 0 и отклонение 1; на test — близко, но не точно:
# так и должно быть, ведь параметры посчитаны не по test
scaled_cols = skewed + symmetric
pd.DataFrame({
    "среднее train": X_train_prep[scaled_cols].mean(),
    "ст.откл. train": X_train_prep[scaled_cols].std(ddof=0),
    "среднее test": X_test_prep[scaled_cols].mean(),
    "ст.откл. test": X_test_prep[scaled_cols].std(ddof=0),
}).round(3)

In [ ]:
# порядковые шкалы и командировки — в диапазоне [0, 1], one-hot — только 0 и 1
other_cols = [c for c in X_train_prep.columns if c not in scaled_cols]
X_train_prep[other_cols].agg(["min", "max"]).T

In [ ]:
# асимметрия скошенных признаков до и после log1p (train)
pd.DataFrame({
    "до": X_train[skewed].skew(),
    "после": X_train_prep[skewed].skew(),
}).round(2)

## 6. Вывод

**Очистка** повторена из части 1: дубликатов и исходных пропусков нет; удалены 3 константных столбца, идентификатор
и 3 противоречивые «ставки»; 244 логически невозможных значения `NumCompaniesWorked` переведены в пропуски.

**Масштабирование `MonthlyIncome`.** Линейные методы (StandardScaler, MinMaxScaler, MaxAbsScaler, RobustScaler) меняют только шкалу,
форма распределения и выбросы остаются. Хуже всего здесь MinMaxScaler: из-за высоких зарплат основная масса сотрудников сжата
в узкую часть отрезка [0, 1]. RobustScaler устойчив к выбросам, но асимметрию не убирает. Лучший вариант для этого признака —
**log1p + StandardScaler**: асимметрия снижается с 1,33 до 0,25, выбросов по правилу 1,5·IQR не остаётся, признак получает среднее 0 и отклонение 1.

**Предобработка всех столбцов** выполнена после разделения 80/20 со стратификацией; все параметры (медиана для пропусков,
средние и отклонения, границы шкал, списки категорий) найдены только на train и применены к test без пересчёта —
утечки данных нет. Итог — 44 числовых признака без пропусков:
- 9 скошенных признаков (асимметрия > 0,75: доход, все показатели стажа, расстояние, число компаний, процент повышения) — log1p + StandardScaler;
- 2 близких к симметричным (`Age`, `TrainingTimesLastYear`) — StandardScaler;
- 9 порядковых шкал — MinMaxScaler в [0, 1] с сохранением порядка;
- `BusinessTravel` — OrdinalEncoder с заданным порядком;
- `Gender`, `OverTime` — один бинарный столбец;
- `Department`, `EducationField`, `JobRole`, `MaritalStatus` — one-hot кодирование;
- целевая `Attrition` — LabelEncoder (No → 0, Yes → 1).

Данные готовы для обучения модели классификации (например, логистической регрессии из плана ПР1).